# Arabic Sentiment Analysis with AraVec, ANN, and LSTM

This notebook explores Arabic tweet representation and sentiment classification using tokenization, Bag of Words, AraVec embeddings, an ANN baseline, and an LSTM sequence model.

## What this notebook demonstrates

- Loading a balanced Arabic sentiment dataset.
- Tokenizing Arabic tweets and removing punctuation.
- Representing text with Bag of Words and AraVec word embeddings.
- Querying semantically similar Arabic words from AraVec.
- Visualizing Arabic word embeddings with PCA.
- Training an ANN classifier using averaged AraVec vectors.
- Training an LSTM model on token sequences.
- Comparing ANN and LSTM performance.



**Very Important:** Please restart the execution environment after executing the code!

From the top menu: **Runtime -> Restart session...**

This is necessary to avoid compatibility issues with the libraries that were just installed or updated.


In [ ]:
"""
print("install/update numpy and gensim...")
!pip install --upgrade --force-reinstall numpy gensim -q
!pip install arabic-reshaper python-bidi -q
print("Installation/update completed.")
"""


In [ ]:
# Concise cell for mounting Google Drive and checking the folder
import os

# Build the full path to the target folder directly

try:
    # Mount directly using the standard path
    drive.mount(drive_mount_point, force_remount=True)
    print(f"Successfully mounted.")

    # Check if the folder exists after mounting
    if os.path.isdir(model_folder_in_drive):
        print("Folder found.")
        try:
            # Display contents for a quick check
            print(f"  Contents: {os.listdir(model_folder_in_drive)}")
        except Exception as e:
            print(f"Error reading folder contents: {e}")
    else:
        print("Folder not found. Please check the path and name.")

except Exception as e:
    print(f"Failed to mount Google Drive or check folder: {e}")


Import all necessary libraries


In [ ]:
import pandas as pd
import numpy as np
import nltk
import gensim
import re
import os
import time
import string
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dropout, Dense
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Download the necessary nltk resources
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)


# 1.1 Read the dataset: Read the dataset (Arabic sentiment dataset)


- first 2500 rows from the positive tweets and the first 2500 rows from the negative tweets.
- were successfully read and merged into a single DataFrame.


In [ ]:
import pandas as pd

positive_file_path = 'train_Arabic_tweets_positive_20190413.tsv'
negative_file_path = 'train_Arabic_tweets_negative_20190413.tsv'

# Variable to store the resulting DataFrame
df_combined = None

# Attempt to read and combine the files

# Read the first 2500 positive rows
df_positive = pd.read_csv(positive_file_path, sep='\t', nrows=2500, header=None, names=['sentiment', 'text'])

# Read the first 2500 negative rows
df_negative = pd.read_csv(negative_file_path, sep='\t', nrows=2500, header=None, names=['sentiment', 'text'])

# Combine the DataFrames
df_combined = pd.concat([df_positive, df_negative], ignore_index=True)

print("Data loaded and merged successfully")


In [ ]:
# Print the dimensions
print(df_combined.shape)
# Print the first 5 rows to see the beginning
print(df_combined.head())
# Print the last 5 rows to see the end
print(df_combined.tail())
# Print a random sample
print(df_combined.sample(5))
# Print a summary of the DataFrame
df_combined.info()


In [ ]:
# Check the distribution of sentiments
if 'sentiment' in df_combined.columns:
    print(df_combined['sentiment'].value_counts())


___


# 1.2 Tokenization: Split the provided text into individual words or tokens


Slicing each tweet in the 'text' column into a list of words (tokens) using the nltk library.

Storing these lists in a new column in the DataFrame named 'tokens'.


In [ ]:
# Create a copy to avoid potential warning
df_combined = df_combined.copy()
df_combined['tokens'] = df_combined['text'].apply(nltk.word_tokenize)
print("'tokens' column added.")

# Display a example for confirmation
print(df_combined['tokens'].iloc[0])


In [ ]:
# Display the first 5 rows of 'text' and 'tokens' columns
print(df_combined[['text', 'tokens']].head())

# Display the updated DataFrame summary
df_combined.info()


___


Apply a cleaning function to remove selected punctuation from the tokens of each tweet, and store the results in a new column.


In [ ]:
punct_set = set(string.punctuation + '،؛؟«»”“ـ')

# Define the cleaning function
def remove_punctuation_from_tokens(tokens_list):
    cleaned_list = []
    for token in tokens_list:
        # The underscore has been replaced with a space, to avoid merging words together.
        token_with_space = token.replace('_', ' ')
        cleaned_token = ''.join(char for char in token_with_space if char not in punct_set)
        if cleaned_token:
            cleaned_list.append(cleaned_token.strip())
    return cleaned_list

# Apply the function to the 'tokens' column and store the result in 'tokens cleaned'
df_combined = df_combined.copy()
df_combined['tokens_cleaned'] = df_combined['tokens'].apply(remove_punctuation_from_tokens)


In [ ]:
# Check punctuation removal
print("Original Tokens: ", df_combined['tokens'].iloc[0])
print("Cleaned Tokens:", df_combined['tokens_cleaned'].iloc[0])

# Display the three columns for comparison
print(df_combined[['text', 'tokens', 'tokens_cleaned']].head())
df_combined.info()


___


# 1.4 Data representation: Use BOW, and Word2vec (Aravec) to represent the textual data in the dataset.


**Data representation: Use BOW**


Create a BOW representation by counting the frequency of unique words and determining the vocabulary size.


In [ ]:
# Join the lists of cleaned tokens into space-separated strings
cleaned_texts = df_combined['tokens_cleaned'].apply(' '.join)

# Create and train CountVectorizer
vectorizer_bow = CountVectorizer()

# Save the matrix and vector for possible later use.
bow_matrix = vectorizer_bow.fit_transform(cleaned_texts)


In [ ]:
# Display results
print(bow_matrix.shape)

# Display a small part of the vocabulary for checking
bow_vocabulary = vectorizer_bow.get_feature_names_out()
vocab_size = len(bow_vocabulary)
print(f"Vocabulary Size {vocab_size}")

# Display a small sample of the vocabulary (first 20 words)
if vocab_size > 0:
    print("\nFirst 20 words in vocabulary:")
    print(bow_vocabulary[:20])


**Data representation: Use Word2vec (Aravec)**


Load the AraVec model


In [ ]:
# Download the AraVec template from Google Drive and define the cleanup function (after restarting and linking)

# Clean/Normalize Arabic Text
def clean_str(text):
    search = ["أ","إ","آ","ة","_","-","/",".","،"," و "," يا ",'"',"ـ","'","ى","\\",'\n', '\t','&quot;','?','؟','!']
    replace = ["ا","ا","ا","ه"," "," ","","",""," و"," يا","","","","ي","",' ', ' ',' ',' ? ',' ؟ ',' ! ']

    #remove tashkeel
    p_tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(p_tashkeel,"", text)

    #remove longation
    p_longation = re.compile(r'(.)\1+')
    subst = r"\1\1"
    text = re.sub(p_longation, subst, text)

    text = text.replace('وو', 'و')
    text = text.replace('يي', 'ي')
    text = text.replace('اا', 'ا')

    for i in range(0, len(search)):
        text = text.replace(search[i], replace[i])

    #trim
    text = text.strip()

    return text

# Define the full path to the model file in Google Drive
# (Ensure Drive is mounted and the path is correct)
aravec_model_path = os.path.join(
)

# Load the model
model_aravec = None
start_time = time.time()

try:
    # Load the model (assumes accompanying .npy file is in the same folder)
    model_aravec = gensim.models.Word2Vec.load(aravec_model_path)
    load_time = time.time() - start_time
    print(f"Model loaded successfully in {load_time:.2f} seconds.")

except FileNotFoundError:
    print(f"Error: File not found at the specified path.")
    print("   Ensure: 1. Drive is mounted. 2. Path is correct. 3. Model files exist.")
except Exception as e:
    print(f"Another error occurred during loading: {e}")


Create a vector representation (AraVec vector representation) for each tweet.


In [ ]:
# Representing tweets using vector averages

# Get the vector size from the model
vector_size = model_aravec.vector_size
print(f"Vector size used from the model: {vector_size}")

# Function to calculate the vector average for a single tweet (list of tokens)
def get_average_vector(tokens_list):
    # Clean up tokens and make sure they are not empty after cleaning
    cleaned_tokens = [clean_str(token) for token in tokens_list if clean_str(token)]

    # Get vectors for words that are only in the vocabulary of the model
    vectors = []
    for token in cleaned_tokens:
        if token in model_aravec.wv: # .wv To access the model's vocabulary and vectors
            vectors.append(model_aravec.wv[token])

    # We return a vector of zeros if we do not find any words from the tweet in the pattern.
    if not vectors:
        return np.zeros(vector_size, dtype=np.float32)

    # Calculate the average
    average_vector = np.mean(vectors, axis=0).astype(np.float32)
    return average_vector

# Apply the function to the 'tokens_cleaned' column to create AraVec vectors
start_time_vec = time.time()
# We use a copy to avoid warnings.
df_combined = df_combined.copy()
# Apply the function and store the result in a new column.
df_combined['aravec_vector'] = df_combined['tokens_cleaned'].apply(get_average_vector)
end_time_vec = time.time()
print(f"AraVec vectors for tweets were created.")
print(f"The operation took: {end_time_vec - start_time_vec:.2f} seconds")


- One vector was created for each of the 5,000 tweets, with each vector having a length of 300.
- The (non-zero) vectors have the correct dimension (300) to match the AraVec model specifications.


In [ ]:
# Number of vectors
print(f"Number of vectors created: {len(df_combined['aravec_vector'])}")

# First 5 vectors
print(df_combined['aravec_vector'].head().apply(lambda x: x.round(4).tolist()))

# # Check that the length of each vector is 300
print(df_combined['aravec_vector'].head().apply(len)) # It should be all 300

# # Check the number of zero vectors
zero_vectors_count = df_combined['aravec_vector'].apply(lambda x: np.all(x == 0)).sum()
print(f"\nNumber of tweets that result in a vector of zeros {zero_vectors_count}")


---


# 1.5 Print the most similar words to “ امتحان “,” الاهلي “،” الاتحاد


- Procedure: Search for the most semantically similar words to the target words ('امتحان', 'الاهلي', 'الاتحاد') after cleaning them.

- A list of the 10 most related words in the vector space learned by the model, along with a similarity score for each word.

- The resulting lists make sense and are consistent with the likely context of these words in the Twitter data on which the model was trained.


In [ ]:
# Find the most similar words

target_words = ["امتحان", "الأهلي", "الاتحاد"]

for word in target_words:
    print(f"\n==========================================")
    print(f"Search for words similar to:'{word}'")

    # Clean the target word using clean_str
    cleaned_word = clean_str(word)
# The word may be empty after cleaning if it is just symbols that have been deleted.
    if not cleaned_word:
          print(f"The word '{word}' is empty after cleanup.")
          continue # Moving on to the next word

    print(f" Word used to search (after cleaning): '{cleaned_word}'")

    # Find similar words in the form
    try:
        # Top 10 most similar words
        similar_words = model_aravec.wv.most_similar(cleaned_word, topn=10)

        print(f"The words most similar to '{cleaned_word}' in the pattern:")
        if similar_words: # Ensure the list is not empty
              for similar_word, score in similar_words:
                  print(f" - {similar_word} (similarity score: {score: .4f})")
        else:
              print(" No similar words found.")

    except KeyError:
        # If the word (after cleaning) is not in the vocabulary of the model
        print(f"The word '{cleaned_word}' is not found in the AraVec model vocabulary.")
    except Exception as e:
        print(f"Another error occurred while searching for similarities to '{cleaned_word}': {e}")


---


# 1.6 Plot and visualize the embedding of the following sentence:
# "الاختبار صعب ، الامتحان كان سهل ، المباراه صعبه الاهلي فاز والاتحاد خسر "


Identifying unique words in the target sentence that have a vector representation (embedding) in the loaded AraVec model (after cleaning)

Extracting these vectors (300 dimensions).

Principal Component Analysis (PCA) was applied to reduce these vectors to only two dimensions for easier graphical display.

- Eleven unique words from the sentence were found in the AraVec model, each with an original vector of 300 dimensions.
- This confirms that the PCA dimensionality reduction process was successful, producing a new matrix with dimensions (11, 2).


In [ ]:
# Target sentence
sentence = "الاختبار صعب ، الامتحان كان سهل ، المباراه صعبه الاهلي فاز والاتحاد خسر"
print(f"Original sentence: {sentence}")

tokens = nltk.word_tokenize(sentence)

words_for_plot = []
vectors_for_plot = []
print("\nWords and vectors to be extracted and drawn:")
for token in tokens:
  # Ignore tokens that are just basic punctuation before cleaning.
  if token in [',', '.', '،']:
      continue
  cleaned_token = clean_str(token)
  # Ensure that the word after cleaning is present in the form and is not empty.
  if cleaned_token and cleaned_token in model_aravec.wv:
      vector = model_aravec.wv[cleaned_token]
      # # Make sure not to add the same word twice (if it is repeated in the sentence)
      if cleaned_token not in words_for_plot:
            words_for_plot.append(cleaned_token)
            vectors_for_plot.append(vector)
            print(f" - {cleaned_token} (Vector shape: {vector.shape})")
  else:
        print(f" - The word '{cleaned_token}' (parent: {token}) does not exist or is empty, it will be ignored.")

# Ensure you have enough vectors for drawing.
if len(vectors_for_plot) < 2:
  print("\nNot enough vectors (less than 2) of sentence words were found in the model to plot.")
else:
  # Dimensionality reduction using PCA to 2D
  pca = PCA(n_components=2, random_state=42) # random_state للتثبيت
  # Convert the vector list to a numpy array
  vectors_matrix = np.array(vectors_for_plot)
  vectors_2d = pca.fit_transform(vectors_matrix)
  print(f"New shape of vectors after PCA: {vectors_2d.shape}")


In [ ]:
# Choose from available styles for drawing.
import matplotlib.pyplot as plt
print(plt.style.available)


The Matplotlib library was used to plot the points representing word vectors after reducing their dimensions to 2 using PCA.

The graph displays a visual representation of the positions of words relative to each other in a two-dimensional space based on their embeddings from the AraVec model. Words that are close together in the graph are assumed to be more semantically similar in the context of the model.


In [ ]:
#chart
try:
    plt.style.use('grayscale') # Choose a drawing style
except: # Fallback style if seaborn is not available fully
      plt.style.use('default')
plt.figure(figsize=(14, 9)) # Drawing size

# رسم النقاط (المتجهات ثنائية البعد)
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], s=100, alpha=0.7, edgecolors='k', c=range(len(words_for_plot)), cmap='viridis') # تلوين النقاط

# Adding word names next to the dots (with Arabic text processing)
for i, word in enumerate(words_for_plot):
# Format and reverse Arabic text for correct display
    reshaped_text = arabic_reshaper.reshape(word)
    bidi_text = get_display(reshaped_text)
    plt.annotate(bidi_text,
                  xy=(vectors_2d[i, 0], vectors_2d[i, 1]),
                  xytext=(5, -5), # Modify displacement
                  textcoords='offset points',
                  ha='right',
                  va='top', # Change vertical alignment
                  fontsize=14, # Enlarge the font
                  bbox=dict(boxstyle='round,pad=0.3', fc='yellow', alpha=0.3)) # Add a background box to the text


plt.title('Sentence Word Vector Visualization using PCA', fontsize=18, weight='bold')
plt.xlabel("First Principal Component (PC1)", fontsize=14)
plt.ylabel("Second Principal Component (PC2)", fontsize=14)
plt.axhline(0, color='grey', lw=0.5) # Horizontal line at zero
plt.axvline(0, color='grey', lw=0.5) # Vertical line at zero
plt.grid(True, linestyle='--', alpha=0.6) # Grid styling

# Show the plot
plt.show()


# 1.1 Build an artificial neural network with the one hidden layer and 4 neurons in the hidden layer and two neurons in the output layer.


Prepare the input matrix X (from AraVec vectors) and the classification matrix y (in one-hot encoding).

These were split into training and test data.

Build a simple neural network (ANN) model using Keras according to the required specifications and compile it.

- Original shape of X and Y (5,000 samples).
- Shapes of training and test data after an 80/20 split (4,000 for training, 1,000 for testing).
- One hidden layer (Hidden_Layer_1) with 4 neurons and an output layer (Output_Layer) with 2 neurons, with the number of parameters calculated.

Note:
UserWarning: This part of the output displays a Keras advisory warning (it does not affect the correctness of the execution)


In [ ]:
# Use np.stack to convert the Series of arrays into a 2D numpy array
X = np.stack(df_combined['aravec_vector'].values)

# Extract Labels (y)
y_labels = df_combined['sentiment'].values

# Convert text labels ('positive', 'negative') to numerical format (0, 1)
label_encoder = LabelEncoder()
y_numerical = label_encoder.fit_transform(y_labels)

# Convert numerical labels to categorical format
y_categorical = to_categorical(y_numerical)

# # Print data information for confirmation
print(f"Shape of feature matrix (X): {X.shape}")
print(f"Shape of one-hot encoded labels (y): {y_categorical.shape}")

# Split Data into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical)

# Print dimensions of training and test data
print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Testing data shape: X={X_test.shape}, y={y_test.shape}")

# Build the ANN Model

# Define input dimension based on AraVec vector size
input_dim = X_train.shape[1] # Should be 300

# # Create a sequential model
model_ann = Sequential(name="Simple_ANN_Sentiment")

# Hidden Layer: 1 layer with 4 neurons
model_ann.add(Dense(units=4, activation='relu', input_shape=(input_dim,), name='Hidden_Layer_1'))

# Output Layer: 2 neurons (for positive/negative)
model_ann.add(Dense(units=2, activation='softmax', name='Output_Layer'))

print("Model built successfully.")

# Compile the Model
model_ann.compile(optimizer='adam',                  # optimizer
                  loss='categorical_crossentropy',   # Suitable for one-hot encoded labels
                  metrics=['accuracy'])              # Monitor accuracy during training

print("Model compiled successfully.")

# Display the model architecture
print("\nModel Summary:")
model_ann.summary()


---


# 1.2 Train an ANN model using the Word2Vec (AraVec) representation.


The training process of the previously built artificial neural network (ANN) model has been executed.


In [ ]:
# rain the Model
print("Starting ANN model training (using AraVec)")

# Define the number of epochs (full training cycles over the data) and batch size
num_epochs = 20  # You can change this number later to experiment with its effect
batch_size = 32  # A common batch size

# Call the fit function to train the model
history_ann = model_ann.fit(X_train, y_train,
                            epochs=num_epochs,
                            batch_size=batch_size,
                            validation_data=(X_test, y_test),
                            verbose=1)
print("ANN model training completed successfully.")


---


# 1.3 Evaluate the ANN on a test set and report accuracy, precision, recall, and F1-score.


The overall accuracy of the model on the test set is approximately 0.68, or 68%.

Precision, recall, and F1-score for each class (negative and positive). These results indicate acceptable performance for the ANN model.


In [ ]:
print("Evaluating ANN model performance on the test set")

# Get model predictions for the test set
predictions_probabilities = model_ann.predict(X_test)

# Convert probabilities to actual class labels (0 or 1)
y_pred_labels = np.argmax(predictions_probabilities, axis=1)

# Get the true labels from the one-hot encoded format
# Convert y_test from [1, 0] or [0, 1] format to 0 or 1
y_true_labels = np.argmax(y_test, axis=1)

# Calculate and print the classification report
print("Classification Report:")
print(classification_report(y_true_labels, y_pred_labels, target_names=label_encoder.classes_))


---


# 1.4 Build an LSTM model with input shape =64, dropout ratio 0.5, and dense layer input shape =2 and activation function= SoftMax.


Converting text data into a sequential numeric format suitable for an LSTM model.



*   Analyzing tweet lengths to determine an appropriate sequence length (maxlens).
*   Using a tokenizer to create a word dictionary and converting each tweet into a sequence of numbers (IDs).

*   Applying padding to make all sequences the same length.

The text data was converted into numeric sequences of equal length and partitioned.


In [ ]:
print("Starting sequence data preparation for LSTM model")

# Analyze sequence lengths and choose maxlen

# Ensure the 'tokens_cleaned' column exists
if 'tokens_cleaned' not in df_combined.columns:
    print("Error: 'tokens_cleaned' column not found. Please ensure previous processing steps were run.")
else:
    # Calculate the length of each token list (each tweet)
    sequence_lengths = df_combined['tokens_cleaned'].apply(len)

    # Display statistics about tweet lengths
    print("\n number of words after cleaning")
    print(sequence_lengths.describe())

    # Choose a value for maxlen (maximum sequence length)
    # can choose a value that covers most lengths
    maxlen = 50
    print(f"\nMaximum sequence length (maxlen) chosen: {maxlen}")

    # Convert words to numbers (Tokenization & Sequencing)

    # Define vocabulary size (number of most frequent words to keep)
    vocab_size = 10000

    # Token for words not in the chosen vocabulary
    oov_tok = "<OOV>"

    # Initialize the Tokenizer
    tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)

    # Build the vocabulary based on the words in our data
    tokenizer.fit_on_texts(df_combined['tokens_cleaned'])

    # Convert lists of tokens to sequences of numbers (IDs)
    X_sequences = tokenizer.texts_to_sequences(df_combined['tokens_cleaned'])
    print(f"\nExample of the first tweet as a token list: {df_combined['tokens_cleaned'].iloc[0]}")
    print(f"Same tweet as a sequence of numbers: {X_sequences[0]}")

    # Padding sequences

    # Apply padding to make all sequences the same length (maxlen)
    X_padded_seq = pad_sequences(X_sequences, maxlen=maxlen, padding='post', truncating='post')
    print(f"\nShape of the sequence matrix after Padding: {X_padded_seq.shape}")
    print(f"Example of the first sequence after Padding:\n{X_padded_seq[0]}")

    # Splitting data using the same method as before
    X_train_seq, X_test_seq, y_train_lstm, y_test_lstm = train_test_split(
        X_padded_seq, y_categorical, # Using the new X_padded_seq and the old y_categorical
        test_size=0.2,
        random_state=42,
        stratify=y_categorical
    )

    print(f"\nShape of LSTM training data: X={X_train_seq.shape}, y={y_train_lstm.shape}")
    print(f"Shape of LSTM test data: X={X_test_seq.shape}, y={y_test_lstm.shape}")
    print("Sequence data preparation for LSTM completed")


Building and Compiling an LSTM Model Using the Keras Sequential API

Start with an Embedding_Layer layer to receive sequences of length 50 and generate embedding vectors of size 128.

*   64-unit LSTM_Layer
*   0.5-unit Dropout_Layer
*   2-unit Dense Output_Layer
*   Softmax Activation Function


In [ ]:
# Define hyperparameters
embedding_dim = 128
lstm_units = 64
dropout_rate = 0.5
output_units = 2

print("\nBuilding LSTM Model (v2)")

# Build the model using the Sequential API with an Input layer
model_lstm_v2 = Sequential([
    Input(shape=(maxlen,), name='Input_Layer'),
    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              # No input_length needed here
              name='Embedding_Layer'),
    LSTM(units=lstm_units, name='LSTM_Layer'), # LSTM layer
    Dropout(rate=dropout_rate, name='Dropout_Layer'), # Dropout layer
    Dense(units=output_units, activation='softmax', name='Output_Layer') # Output layer
], name="LSTM_Sentiment_Model_v2")

print("Model structure built successfully (v2).")

# Compile the model
print("\nCompiling LSTM model (v2)...")
model_lstm_v2.compile(optimizer='adam',
                      loss='categorical_crossentropy', # Loss function for one-hot encoded labels
                      metrics=['accuracy'])           # Metric to monitor
print("Model compiled successfully (v2).")

# Display the model summary
print("\nLSTM Model Summary (v2):")
model_lstm_v2.summary()


---


# 1.5 Compile and Train the model on the training set.


The training process of the pre-built LSTM model has been implemented.


In [ ]:
# Train the LSTM Model
print("\nStarting LSTM model training")

# Define number of epochs and batch size
num_epochs_lstm = 10 # Suggested number of training epochs
batch_size_lstm = 32 # Batch size

# Train the model
# # Use sequential training and test data
history_lstm = model_lstm_v2.fit(X_train_seq, y_train_lstm,
                                 epochs=num_epochs_lstm,
                                 batch_size=batch_size_lstm,
                                 validation_data=(X_test_seq, y_test_lstm), # Use test data for validation
                                 verbose=1) # Display progress details

print("\nLSTM model training completed successfully")


---


# 1.6 Evaluate the classifier on a test set and report accuracy, precision, recall, and F1-score.


- The LSTM model performed significantly better than a simple ANN model on this sentiment analysis task.

- This underscores the importance of LSTM's ability to understand word sequences and context in a text, something that vector-based ANN models lack.

- 83% accuracy is a very good result for this type of model on this dataset.


In [ ]:
# Evaluate the LSTM Classifier on the Test Set
print("\nEvaluating LSTM model performance on the test set")

# Get LSTM model predictions for the sequence test set
predictions_probabilities_lstm = model_lstm_v2.predict(X_test_seq, verbose=0)

# Convert probabilities to actual class labels (0 or 1)
y_pred_labels_lstm = np.argmax(predictions_probabilities_lstm, axis=1)

# Get the true labels from the one-hot encoded format
y_true_labels_lstm = np.argmax(y_test_lstm, axis=1)

# Calculate and print the classification report for the LSTM model
print("Classification Report for LSTM Model:")
print(classification_report(y_true_labels_lstm, y_pred_labels_lstm, target_names=label_encoder.classes_))


---


# 1.7 Discuss the results and write your insights about the difference between the two models.


*   The results showed a significant and clear difference in performance between the two models on the same test set.

*   Simple ANN model: achieved an overall accuracy of approximately 68%. The Precision, Recall, and F1-score metrics were also modest, around the same level (0.67-0.68).

*   LSTM model: achieved an overall accuracy of approximately 83%. The Precision, Recall, and F1-score metrics were significantly higher (around 0.75-0.91).


*   Reason: LSTM's ability to understand the sequence of words and their context within a tweet.
*   LSTM has a "memory" for processing text step by step.
*   The ANN model, by relying on average word vectors (AraVec), lost word order information and treated the tweet as a "bag of words," limiting its ability to understand nuances of meaning.


*   Using a learnable embedding layer with LSTMs allowed the model to generate word representations tailored to the sentiment analysis task, potentially better than pre-prepared generic vectors.

*   The results highlight that models designed to understand sequence and context ,such as LSTMs, are significantly more effective for complex NLP tasks ,such as sentiment analysis, than simpler models that ignore these important aspects.
